# RecSys: Домашнее задание 1

Amazon Reviews 2014 - Video Games, 5-core.
Рекомендуем пользователю ещё не оценённые товары. Взаимодействие - отзыв с оценкой.
Релевантным считаем товар с оценкой не ниже 4.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import data_utils as data

np.random.seed(42)


## 1. Данные и подготовка

Video Games 5-core содержит около 230 тысяч отзывов. В исходной выборке у каждого пользователя
и товара не менее пяти отзывов. Используем всю выборку.

Поля: `reviewerID` - пользователь, `asin` - товар, `overall` - оценка от 1 до 5,
`unixReviewTime` - время отзыва в секундах Unix. В метаданных используем название,
бренд и категории; связь по `asin`.


In [ ]:
reviews_path = data.download(data.REVIEWS_URL, "data/reviews_Video_Games_5.json.gz")
metadata_path = data.download(data.METADATA_URL, "data/meta_Video_Games.json.gz")
raw = data.load_reviews(reviews_path)
raw.head()


Проверяем пропуски, дубликаты, активность пользователей, популярность товаров,
разреженность и метаданные. Удаляем записи с некорректными ID, оценками или временем.
Для повторов пользователь-товар оставляем первое наблюдение, чтобы поздняя оценка
не заменила доступную в прошлом. Низкие оценки сохраняем. Товары без метаданных
не удаляем, дополнительных фильтров активности не применяем.


In [ ]:
raw.isna().sum()


In [ ]:
raw[["rating", "timestamp"]].describe()


In [ ]:
print("Полные дубликаты:", raw.duplicated().sum())
print("Повторы пользователь-товар:", raw.duplicated(["user", "item"]).sum())


In [ ]:
interactions, cleaning_report = data.clean_reviews(raw)
display(cleaning_report)

display(pd.DataFrame([
    data.describe_interactions(raw),
    data.describe_interactions(interactions),
], index=["raw", "clean"]))

assert 100_000 <= len(interactions) <= 2_000_000
assert interactions.user.nunique() >= 1_000
assert interactions.item.nunique() >= 1_000


In [ ]:
activity = interactions.groupby("user").size()
popularity = interactions.groupby("item").size()
display(pd.DataFrame({"user_activity": activity.describe(),
                      "item_popularity": popularity.describe()}))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(activity, bins=50, log=True)
axes[0].set(title="Активность пользователей", xlabel="Число отзывов", ylabel="Пользователи (log)")
axes[1].hist(popularity, bins=50, log=True)
axes[1].set(title="Популярность товаров", xlabel="Число отзывов", ylabel="Товары (log)")
interactions.rating.value_counts().sort_index().plot.bar(ax=axes[2], rot=0)
axes[2].set(title="Оценки", xlabel="Оценка", ylabel="Число отзывов")
plt.tight_layout()
plt.show()

dates = pd.to_datetime(interactions.timestamp, unit="s", utc=True)
interactions.assign(month=dates.dt.to_period("M").astype(str)).groupby("month").size().plot(
    figsize=(13, 3), title="Взаимодействия по месяцам", ylabel="Отзывы", xlabel="Месяц")
plt.show()
print("Временной диапазон UTC:", dates.min(), "-", dates.max())


In [ ]:
from collections import Counter

metadata = data.load_metadata(metadata_path, interactions.item.unique())
catalog = pd.DataFrame({"item": sorted(interactions.item.unique())}).merge(metadata, on="item", how="left")
metadata_report = pd.Series({
    "items": len(catalog),
    "without_metadata": int((~catalog.item.isin(metadata.item)).sum()),
    "without_title": int(catalog.title.fillna("").eq("").sum()),
    "without_brand": int(catalog.brand.fillna("").eq("").sum()),
    "without_categories": int(catalog.categories.map(lambda x: not isinstance(x, list) or not x).sum()),
})
display(metadata_report.to_frame("count"))
display(catalog.head())
category_counts = Counter()
for paths in metadata.categories:

    leaves = {path[-1] for path in paths if isinstance(path, list) and path}
    category_counts.update(leaves)
if category_counts:
    pd.Series(dict(category_counts.most_common(12))).sort_values().plot.barh(
        figsize=(9, 5), title="Частые конечные категории", xlabel="Товары")
    plt.show()


## 2. Оценка качества

Разбиваем по времени: примерно 80% взаимодействий - train, 10% - validation,
10% - test. Общие временные границы исключают обучение на будущих событиях;
одинаковые timestamp остаются в одной части.

Кандидаты - все товары train. Новых пользователей и товары исключаем из оценки,
их количество выводим отдельно. Оцениваем пользователей хотя бы с одним релевантным
товаром в соответствующей части. Уже оценённые товары исключаем независимо от оценки.
Для validation известная история - train, для test - train и validation.
Модель и каталог фиксируем по train, перед test не переобучаем.
Этот протокол используем для всех моделей.

Основная метрика - **NDCG@10**: учитывает позиции релевантных товаров.
Дополнительная - **Recall@10**: доля найденных релевантных товаров.
Усредняем по пользователям.


In [ ]:
K = 10
RELEVANCE_THRESHOLD = 4
TRAIN_FRACTION = 0.8
VALIDATION_FRACTION = 0.1
BATCH_SIZE = 128

train, validation, test, boundaries = data.temporal_split(
    interactions, TRAIN_FRACTION, VALIDATION_FRACTION)
assert train.timestamp.max() < validation.timestamp.min()
assert validation.timestamp.max() < test.timestamp.min()
assert len(train) + len(validation) + len(test) == len(interactions)
display(pd.DataFrame([
    {"split": name, **data.describe_interactions(frame),
     "start_utc": pd.to_datetime(frame.timestamp.min(), unit="s", utc=True),
     "end_utc": pd.to_datetime(frame.timestamp.max(), unit="s", utc=True)}
    for name, frame in [("train", train), ("validation", validation), ("test", test)]
]))

users, items = data.make_mappings(train)
X_positive = data.interaction_matrix(train, users, items, RELEVANCE_THRESHOLD)
X_seen = data.interaction_matrix(train, users, items)
validation_targets, validation_coverage = data.evaluation_targets(
    validation, users, items, X_seen, RELEVANCE_THRESHOLD)
display(pd.Series(validation_coverage).to_frame("validation"))
assert validation_targets, "Нет пользователей для оценки: проверьте разбиение"
print("Матрица train:", X_positive.shape, "; положительных пар:", X_positive.nnz)
print("Пользователей без положительной истории train:", int((X_positive.getnnz(axis=1) == 0).sum()))


## 3. Item-based CF

Положительная история: $X_{ui}=1$, если в train есть оценка $r_{ui}\ge4$, иначе 0.
$n_{ij}=(X^TX)_{ij}$ - число общих пользователей товаров,
$n_i=\sum_u X_{ui}$. Используем косинусное сходство со сглаживанием:

$$
s_{ij}=\frac{n_{ij}}{\sqrt{n_i n_j}}\frac{n_{ij}}{n_{ij}+\lambda},\qquad s_{ii}=0.
$$

Без общих пользователей сходство равно нулю; при $\lambda=0$ сглаживания нет.
Сглаживание уменьшает влияние случайных совпадений. Для каждого товара сохраняем
$L$ соседей с наибольшим сходством в матрице $W$.
Баллы рекомендаций:

$$
\hat y_{uj}=(XW)_{uj}=\sum_{i:X_{ui}=1}W_{ij}.
$$

Выбираем 10 неизвестных товаров с максимальным баллом. При равенстве баллов
приоритет у более популярных в положительном train товаров, затем у меньшего ID.
Для пользователей без положительной истории получается рекомендация по популярности.


### Подбор параметров

`n_neighbors ∈ {50, 100, 200}`, `shrinkage ∈ {0, 10}`.
Выбираем максимум NDCG@10 на validation; при равенстве - меньше соседей,
затем меньше `shrinkage`. Размер блока - 256, пакет оценки - 128 пользователей.


In [ ]:
import json
from pathlib import Path
from itertools import product

from models.item_knn import ItemKNN
from evaluation import evaluate

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

search_space = {"n_neighbors": [50, 100, 200], "shrinkage": [0.0, 10.0]}
validation_rows = []
for neighbors, shrinkage in product(search_space["n_neighbors"], search_space["shrinkage"]):
    candidate = ItemKNN(n_neighbors=neighbors, shrinkage=shrinkage, block_size=256)
    candidate.fit(X_positive)
    scores = evaluate(candidate, validation_targets, X_seen, k=K, batch_size=BATCH_SIZE)
    row = {"model": "ItemKNN", "n_neighbors": neighbors, "shrinkage": shrinkage,
           **scores}
    validation_rows.append(row)
    pd.DataFrame(validation_rows).to_csv(RESULTS_DIR / "item_knn_validation.csv", index=False)
    print(row)
    del candidate

validation_results = pd.DataFrame(validation_rows).sort_values(
    ["NDCG@10", "n_neighbors", "shrinkage"], ascending=[False, True, True], kind="stable")
display(validation_results)
best_row = validation_results.iloc[0]
best_params = {"n_neighbors": int(best_row.n_neighbors), "shrinkage": float(best_row.shrinkage)}
(RESULTS_DIR / "item_knn_best_params.json").write_text(
    json.dumps(best_params, indent=2), encoding="utf-8")
print("Выбрано по validation NDCG@10:", best_params)
best_model = ItemKNN(**best_params).fit(X_positive)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for shrinkage, group in validation_results.groupby("shrinkage"):
    group = group.sort_values("n_neighbors")
    for ax, metric in zip(axes, ["NDCG@10", "Recall@10"]):
        ax.plot(group.n_neighbors, group[metric], marker="o", label=f"shrinkage={shrinkage:g}")
        ax.set(xlabel="Число соседей", ylabel=metric, title=f"Validation: {metric}")
        ax.legend()
plt.tight_layout()
plt.show()

unsmoothed = validation_results[validation_results.shrinkage.eq(0)].iloc[0]
smoothed = validation_results[validation_results.shrinkage.eq(10)].iloc[0]
print(f"Лучший validation NDCG@10: {best_row['NDCG@10']:.5f}; "
      f"Recall@10: {best_row['Recall@10']:.5f}")
print(f"Разница лучших NDCG@10 со сглаживанием и без: "
      f"{smoothed['NDCG@10'] - unsmoothed['NDCG@10']:+.5f}")
